# Deep Learning using Convolutional Neural Networks to Classify Defects on the WM-811K Dataset

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns


#### Import and inspect data

In [31]:
df = pd.read_pickle('../data/LSWMD.pkl')

In [32]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 811457 entries, 0 to 811456
Data columns (total 6 columns):
 #   Column          Non-Null Count   Dtype  
---  ------          --------------   -----  
 0   waferMap        811457 non-null  object 
 1   dieSize         811457 non-null  float64
 2   lotName         811457 non-null  object 
 3   waferIndex      811457 non-null  float64
 4   trianTestLabel  811457 non-null  object 
 5   failureType     811457 non-null  object 
dtypes: float64(2), object(4)
memory usage: 37.1+ MB


In [33]:
df.rename(columns={"trianTestLabel": "trainTestLabel"}, inplace=True)

In [34]:
non_numeric_cols = df.columns[df.dtypes == 'object'].to_list()
non_numeric_cols.remove("waferMap")
print(non_numeric_cols)

def unpack(x):
    arr = np.asarray(x)

    if arr.size == 0:
        return None

    return arr[0][0]

['lotName', 'trainTestLabel', 'failureType']


In [35]:
df["failureType"] = df["failureType"].apply(unpack)
df["trainTestLabel"] = df["trainTestLabel"].apply(unpack)
df["failureType"].value_counts(dropna=False)

failureType
NaN          638507
none         147431
Edge-Ring      9680
Edge-Loc       5189
Center         4294
Loc            3593
Scratch        1193
Random          866
Donut           555
Near-full       149
Name: count, dtype: int64

In [36]:
df["trainTestLabel"].value_counts(dropna=False)

trainTestLabel
NaN         638507
Test        118595
Training     54355
Name: count, dtype: int64

In [37]:
print(f'Number of wafermaps: {df["waferMap"].shape}')
print(f'Shape of wafermaps: {df["waferMap"][0].shape}')

Number of wafermaps: (811457,)
Shape of wafermaps: (45, 48)


In [38]:
labeled_df = df[df["trainTestLabel"].notna().copy()]
labeled_df.shape

(172950, 6)

In [40]:
labeled_df["failureType"].value_counts(dropna=False)

failureType
none         147431
Edge-Ring      9680
Edge-Loc       5189
Center         4294
Loc            3593
Scratch        1193
Random          866
Donut           555
Near-full       149
Name: count, dtype: int64

In [41]:
wm_df = labeled_df[labeled_df["failureType"] != 'none'].copy()
wm_df["failureType"].value_counts()

failureType
Edge-Ring    9680
Edge-Loc     5189
Center       4294
Loc          3593
Scratch      1193
Random        866
Donut         555
Near-full     149
Name: count, dtype: int64

#### Visualisations

In [ ]:
# Class distribution -- labeled wafers only
counts = labeled_df['failureType'].value_counts().sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(10, 4))
bars = ax.bar(counts.index, counts.values,
              color=plt.cm.viridis([i / len(counts) for i in range(len(counts))]))
ax.set_title('Class distribution -- labeled wafers (train + test)', fontsize=13)
ax.set_xlabel('Failure type')
ax.set_ylabel('Count')
ax.tick_params(axis='x', rotation=30)
for bar, val in zip(bars, counts.values):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 500,
            f'{val:,}', ha='center', va='bottom', fontsize=8)
plt.tight_layout()
plt.show()


#### Sample wafer maps by defect type

Each wafer map is a 45x48 array where pixel values encode die status:
- **0** -- not tested / outside die
- **1** -- pass
- **2** -- fail

Showing one randomly-chosen example per class (including `none`).

In [ ]:
classes = labeled_df['failureType'].unique()
classes = ['none'] + sorted([c for c in classes if c != 'none'])

n_cols = 3
n_rows = -(-len(classes) // n_cols)

cmap = plt.cm.get_cmap('RdYlGn_r', 3)

fig, axes = plt.subplots(n_rows, n_cols, figsize=(n_cols * 4, n_rows * 3.5))
axes = axes.flatten()

rng = np.random.default_rng(42)
for ax, cls in zip(axes, classes):
    subset = labeled_df[labeled_df['failureType'] == cls]
    sample = subset.iloc[rng.integers(len(subset))]
    wmap = np.asarray(sample['waferMap'], dtype=float)
    ax.imshow(wmap, cmap=cmap, vmin=0, vmax=2, interpolation='nearest')
    ax.set_title(f'{cls}\n(n={len(subset):,})', fontsize=10)
    ax.axis('off')

for ax in axes[len(classes):]:
    ax.set_visible(False)

legend_patches = [
    mpatches.Patch(color=cmap(0), label='0 -- outside / untested'),
    mpatches.Patch(color=cmap(1), label='1 -- pass'),
    mpatches.Patch(color=cmap(2), label='2 -- fail'),
]
fig.legend(handles=legend_patches, loc='lower center', ncol=3,
           bbox_to_anchor=(0.5, -0.02), fontsize=10)

fig.suptitle('Sample wafer maps by defect type', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()


#### Wafer map size distribution

Confirming whether all labeled wafer maps share the same dimensions -- important for fixed-size CNN input.

In [ ]:
shapes = labeled_df['waferMap'].apply(lambda w: np.asarray(w).shape)
shape_counts = shapes.value_counts()
print('Unique wafer map shapes in labeled set:')
print(shape_counts.to_string())
print(f'\nAll same shape: {shape_counts.shape[0] == 1}')
